# Supervised Regression mit `scikit-learn`

## Schritt 1: Werkzeuge laden

Ladet folgende Werkzeuge in eurem Python-Kernel:

* `read_csv` aus `pandas`
* `train_test_split` aus dem `model_selection`-Modul in `scikit-learn`
* `DecisionTreeRegressor` aus dem `tree`-Modul in `scikit-learn`
* `LinearRegression` aus dem `linear_model`-Modul in `scikit-learn`
* `PolynomialFeatures` aus dem `preprocessing`-Modul in `scikit-learn`
* `r2_score` aus dem `metrics`-Modul in `scikit-learn`
* `mean_absolute_error` aus dem `metrics`-Modul in `scikit-learn`
* `root_mean_squared_error` aus dem `metrics`-Modul in `scikit-learn`

In [19]:
from pandas import read_csv
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

In [20]:
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

## Schritt 2: Daten Laden

Ladet den `bikesharing.csv`-Datensatz in `pandas`. Bereinigt den Datensatz beim Laden, falls nötig.

In [21]:
data = read_csv(
    "https://raw.githubusercontent.com/mckoh/data-science-mcit-ba-im-2026/refs/heads/main/data/bikesharing.csv",
    sep=";",
    skip_blank_lines=True,
    skiprows=1,
    usecols=range(16)
)

In [22]:
data = data.loc[data["cnt"].notna()]

In [23]:
X = data[[
    'season',
    'holiday',
    'weekday',
    'workingday',
    'weathersit',
    'temp',
    'atemp',
    'hum',
    'windspeed',
    'month'
]]

y = data['cnt']

## Schritt 3: Daten explorieren

Schaut euch die Daten an. Verwendet dazu Statistiken, Plots und die Standard-Methoden, die `pandas` für euch bereithält.

In [24]:
data.info()

<class 'pandas.DataFrame'>
Index: 307 entries, 0 to 307
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          307 non-null    float64
 1   season      307 non-null    str    
 2   holiday     307 non-null    str    
 3   weekday     307 non-null    str    
 4   workingday  307 non-null    str    
 5   weathersit  307 non-null    str    
 6   temp        306 non-null    float64
 7   atemp       306 non-null    float64
 8   hum         303 non-null    float64
 9   windspeed   304 non-null    float64
 10  casual      307 non-null    float64
 11  registered  307 non-null    float64
 12  cnt         307 non-null    float64
 13  day         307 non-null    float64
 14  month       307 non-null    float64
 15  year        307 non-null    float64
dtypes: float64(11), str(5)
memory usage: 40.8 KB


In [25]:
print(X.shape, y.shape)

(307, 10) (307,)


## Schritt 4: Daten vorbereiten

Bereitet die Daten so vor, dass ihr sie anschließend zum Machine Learning verwenden könnt.

In [26]:
numerical_cols = make_column_selector(dtype_include=['number'])
categorical_cols = make_column_selector(dtype_exclude=['number'])

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

In [27]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [28]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [29]:
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(245, 22) (245,)
(62, 22) (62,)


## Schritt 5: Machine Learning

Verwendet den `DecisionTreeRegressor` und die `LinearRegression` um zwei Regressionsmodelle zu entwickeln. Verwendet diese Modelle anschließend um für eure Daten passende Vorhersagen zu erstellen. Nutzt diese Vorhersagen anschließend, um die Qualität eurer Modelle zu überprüfen. Verwendet dazu `r2_score`, `mean_absolute_error` und `root_mean_squared_error`.

In [43]:
tree_regressor = DecisionTreeRegressor(max_depth=3)
tree_regressor.fit(X_train, y_train)
tree_predictions = tree_regressor.predict(X_test)

In [44]:
linear_regressor = LinearRegression()
linear_regressor.fit(X_train, y_train)
linear_predictions = linear_regressor.predict(X_test)

In [45]:
print("Decision Tree R^2:", r2_score(y_test, tree_predictions))
print("Decision Tree MAE:", mean_absolute_error(y_test, tree_predictions))
print("Decision Tree RMSE:", root_mean_squared_error(y_test, tree_predictions))

Decision Tree R^2: 0.707889670799377
Decision Tree MAE: 577.570186891961
Decision Tree RMSE: 837.8571445829253


In [46]:
print("Linear Regression R^2:", r2_score(y_test, linear_predictions))
print("Linear Regression MAE:", mean_absolute_error(y_test, linear_predictions))
print("Linear Regression RMSE:", root_mean_squared_error(y_test, linear_predictions))

Linear Regression R^2: 0.7593729325829743
Linear Regression MAE: 547.3473505618068
Linear Regression RMSE: 760.4466645908774


## Extra-Schritt: Polynomialregression

Erstellt ein drittes Modell und verwendet dabei die Klasse `PolynomialFeatures` um die `LinearRegression` auf ein höheres Polynomiallevel zu heben. Verwendet Gemini/ChatGPT/Copilot/... um euch dabei beraten zu lassen, wie ihr so etwas umsetzen könnt. Vergleicht dieses Modell anschließend mit eurer originalen linearen Regression.

In [37]:
degree = 3

X_train_poly = PolynomialFeatures(degree=degree).fit_transform(X_train)
X_test_poly = PolynomialFeatures(degree=degree).fit_transform(X_test)

In [38]:
poly_regressor = LinearRegression()
poly_regressor.fit(X_train_poly, y_train)
poly_predictions = poly_regressor.predict(X_test_poly)

In [39]:
print("Linear Regression R^2:", r2_score(y_test, poly_predictions))
print("Linear Regression MAE:", mean_absolute_error(y_test, poly_predictions))
print("Linear Regression RMSE:", root_mean_squared_error(y_test, poly_predictions))

Linear Regression R^2: 0.35081888278443185
Linear Regression MAE: 945.4238895667801
Linear Regression RMSE: 1249.0489531919727
